# Drug–target interaction prediction from 3D structures (EGNN + DrugBAN)

Predicts whether a compound binds a protein from the 3D structure of both. This is **version 2**: the
encoders are rewritten and the drug/protein features are much richer than in version 1. What changed
and why is listed at the end of this cell.

| | Structure | Graph nodes | Graph edges | Node features |
|---|---|---|---|---|
| **Drug** | RDKit 3D conformer | heavy atoms | all atom pairs (bond type + bond stereo, or "no bond") | element, degree, charge, aromatic, hybridization, H count, in ring, **chiral tag** |
| **Protein** | AlphaFold 3 predicted structure | Cα atoms (one per residue) | 16 nearest residues in space (sequence separation) | **9-residue sequence window**, **local backbone geometry**, pLDDT (RBF-encoded) |

Each graph is encoded by an **EGNN** (Satorras et al. 2021), so the prediction does not change when a
molecule is rotated or translated. The two sets of node features are combined by **DrugBAN's bilinear
attention network** and **MLP decoder** (Bai et al. 2023), with one deliberate change: padded positions
are now masked out (see below). The decoder outputs the binding probability.

**Data.** Everything comes from the Hugging Face dataset [`Xingqiao-2002/DTI`](https://huggingface.co/datasets/Xingqiao-2002/DTI),
which has the BindingDB, BioSNAP and Human benchmarks with DrugBAN's original splits. The notebook
downloads only what one run needs (about 50 MB):

- `main/<dataset>/<split>/*.csv`: drug–protein pairs with columns `SMILES, Protein, Y` (`Y` = 1 if they interact)
- `main/drug_sdf/drug_conformers.sdf.gz`: one 3D conformer per compound
- `main/protein_af3/protein_ca.npz`: Cα coordinates and pLDDT of every AlphaFold 3 structure

| Dataset | Splits |
|---|---|
| `bindingdb` | `random`, `cluster` |
| `biosnap` | `random`, `cluster` |
| `human` | `random`, `cold` |

**How to run.** Choose the dataset and split in the configuration cell, then run all cells. On Colab the
setup cell installs `rdkit` and mounts Google Drive; elsewhere, install
`torch rdkit scikit-learn pandas huggingface_hub tqdm` first. Results go to `runs/<dataset>_<split>/`
(`best.pt`, `history.csv`, `results.json`) — under `MyDrive/dti/` on Colab, under the current directory
otherwise. Set `max_train_pairs` to a small number for a quick end-to-end smoke test.

---

### What changed from version 1, and why

**1. Padded positions are masked in the bilinear attention (the most important fix).**
`BANLayer` pools over *every* (atom, residue) pair with `einsum`, and `FCNet` ends in `Linear + ReLU`
with a bias — so a zero-padded position does **not** produce a zero vector, and each one adds a constant
to the fused feature. Measured on the untrained layer, growing the protein from 465 to 3665 padded
positions changes the fused feature's norm from 4.6e4 to 8.7e4 and drops its cosine similarity to the
unpadded version to 0.82. DrugBAN pads to a *fixed* 1200/290, so its offset is constant and learnable;
here the padding was the batch maximum, which ranges from 18 to 3661 residues — so the same pair got
different features depending on which proteins happened to share its batch, and training (`shuffle=True`)
and evaluation (`shuffle=False`) saw different distributions. `to_dense` already returned the mask; it
was being discarded.

**2. Batches are grouped by protein length.** With random batching, a batch of 32 is padded to 1759
residues on average while the true mean is 540 — **226% of the compute is spent on padding**. Sorting
similar lengths together removes almost all of it, so epochs are roughly 3x faster *and* the masking
above has less to do.

**3. Proteins are truncated to 1200 residues** (`max_res`), as DrugBAN does. This affects 7.9% of
proteins and keeps 92.9% of all residues.

**4. The protein encoder gets real features.** Version 1 gave each residue only its own amino-acid type
plus pLDDT, with sequence information surviving only as an 8-bin `|i-j|` edge label. DrugBAN's protein
side is three 1D convolutions (kernels 3/6/9, so a ~16-residue window). Now each residue sees a
**9-residue sequence window** (i-4 … i+4) and **local backbone geometry**: Cα–Cα distances to i±1 and
i±2, the pseudo bond angle, and the *signed* pseudo dihedral. pLDDT is RBF-encoded instead of a scalar
fed through `Linear(1, dim)`.

**5. The drug encoder knows about chirality.** An EGNN that sees only distances is invariant to
reflection, so version 1 could not tell enantiomers apart — and 148 of 300 sampled conformers have at
least one chiral centre. Atom features now include the RDKit chiral tag (recovered with
`AssignStereochemistryFrom3D`, since the SDF stores coordinates only) and edges carry bond stereo.
Atom features are concatenated through an MLP rather than summed.

**6. Mixed precision and an in-epoch progress bar.** `amp=True` uses bf16 where supported, else fp16
with a gradient scaler. The model is unchanged by this; turn it off to compare numerics.

Because of the signed dihedral, the model is **no longer invariant to reflection — on purpose**. Proteins
are chiral (an α-helix is right-handed), so a mirrored structure *should* score differently. The
self-check at the end of the model cell reports both numbers.

The AlphaFold 3 structures are subject to the [AlphaFold 3 Output Terms of Use](https://github.com/google-deepmind/alphafold3/blob/main/OUTPUT_TERMS_OF_USE.md).

**Version 6** keeps the version 5 drug encoder and applies the same idea to the protein: edges are peptide neighbours plus Ca pairs within 8 Å, and the attention sees the EGNN state concatenated with the per-residue features. The training loop is unchanged.


**Version 8** keeps those graphs and the same splits. Both encoders use dim 256, the protein EGNN has 2 layers, and protein edges are dropped with probability 0.2 during training.


In [ ]:
# ===================== 0. Colab setup (does nothing outside Colab) =====================
import sys
if 'google.colab' in sys.modules:
    !pip -q install rdkit
    from google.colab import drive
    drive.mount('/content/drive')      # results are saved to MyDrive/dti/runs/

In [ ]:
# ===================== 1. Configuration and data download =====================
import copy, gc, gzip, hashlib, json, math, os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import snapshot_download
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix

RDLogger.DisableLog('rdApp.*')

CFG = dict(
    dataset='biosnap',      # 'bindingdb' | 'biosnap' | 'human'
    split='random',         # bindingdb, biosnap: 'random' | 'cluster';   human: 'random' | 'cold'
    epochs=100, batch=32, lr=2e-4, wd=1e-4, warmup_epochs=1, patience=15,  # training
    dim=256, drug_layers=4, prot_layers=2, emb=32, edim=24,                # model size
    max_res=1200,           # truncate proteins to this many residues (DrugBAN uses 1200)
    bucket=True,            # group batches by protein length, so padding stays small
    bucket_pool=32,         # batches per shuffled pool; larger = less padding, less batch randomness
    amp=True,               # mixed precision on CUDA (bf16 where supported, else fp16 + GradScaler)
    mask_pad=True,          # mask padded positions in the bilinear attention (set False for v1 behaviour)
    seed=42,
    # DataLoader processes (0 = load batches in the main process). os.cpu_count() reports the whole
    # machine, not this job's share, so prefer the Slurm allocation when there is one.
    workers=max(0, min(4, int(os.environ.get('SLURM_CPUS_PER_TASK') or os.cpu_count() or 1) - 1)),
    max_train_pairs=None,   # set to a small number (e.g. 64) for a quick smoke test; also caps val/test
)
ROOT = '/content/drive/MyDrive/dti' if os.path.isdir('/content/drive/MyDrive') else '.'
OUT = f"{ROOT}/runs/{CFG['dataset']}_{CFG['split']}"
os.makedirs(OUT, exist_ok=True)
torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True

# Download only the files this run needs (public dataset, no login). Files already present are reused.
# On Colab they go to /content/DTI, visible in the Files panel; elsewhere to the Hugging Face cache.
DATA = snapshot_download('Xingqiao-2002/DTI', repo_type='dataset',
                         local_dir='/content/DTI' if os.path.isdir('/content') else None, allow_patterns=[
    f"main/{CFG['dataset']}/{CFG['split']}/*",     # pairs: SMILES, Protein, Y
    'main/drug_sdf/drug_conformers.sdf.gz',        # one RDKit 3D conformer per compound
    'main/protein_af3/protein_ca.npz',             # Ca coordinates + pLDDT of every AF3 structure
])
print('Data downloaded to', DATA, '| results in', OUT, '| device:', dev)

# Compounds and proteins are identified by the first 16 hex digits of the md5 of their SMILES / sequence.
md5 = lambda s: hashlib.md5(s.encode()).hexdigest()[:16]

## 2. Drug graph

Each compound has one RDKit conformer (ETKDGv3 + MMFF94, largest fragment only). Hydrogens are removed,
every heavy atom becomes a node. Edges are the chemical bonds plus every other pair within 5 Å.
The EGNN reads each edge's 3D distance and which bond joins its atoms (none, single, double, triple,
aromatic), together with that bond's stereochemistry (E/Z, cis/trans).

**Chirality.** The EGNN's messages depend only on distances, which are the same for a molecule and its
mirror image — so geometry alone cannot distinguish enantiomers. The SDF file stores 3D coordinates but
no stereo flags, so `AssignStereochemistryFrom3D` recovers them and each atom's chiral tag (CW / CCW /
unspecified) goes in as a categorical feature. 148 of 300 sampled conformers have at least one chiral
centre, so this is not a rare case.

In [ ]:
# ===================== 2. Drug: RDKit conformer -> graph =====================
ELEM = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'I', 'P', 'B', 'Si', 'Se']
HYB = [Chem.HybridizationType.SP, Chem.HybridizationType.SP2, Chem.HybridizationType.SP3,
       Chem.HybridizationType.SP3D, Chem.HybridizationType.SP3D2]
BOND = {Chem.BondType.SINGLE: 1, Chem.BondType.DOUBLE: 2, Chem.BondType.TRIPLE: 3, Chem.BondType.AROMATIC: 4}
N_BOND, N_STEREO, N_CHIRAL = 5, 6, 4      # no bond/single/double/triple/aromatic; bond stereo; chiral tag
# element, degree, formal charge, aromatic, hybridization, num H, in ring, chiral tag
ATOM_VOCAB = [len(ELEM) + 1, 7, 5, 2, len(HYB) + 1, 5, 2, N_CHIRAL]


def load_drug_conformers(path, keep=None):
    """All conformers are in one SDF file; each molecule's name is its id (md5 of the SMILES).
    `keep` restricts the result to those ids -- one split needs a few thousand of the 21832 molecules,
    and skipping the rest saves both time and a couple of GB of RAM."""
    with gzip.open(path) as fh:
        return {m.GetProp('_Name'): m for m in Chem.ForwardSDMolSupplier(fh, removeHs=True)
                if m is not None and (keep is None or m.GetProp('_Name') in keep)}


def drug_graph(mol):
    # The SDF carries coordinates but no stereo flags, so derive them from the 3D conformer first.
    Chem.AssignStereochemistryFrom3D(mol)
    pos = torch.tensor(mol.GetConformer().GetPositions(), dtype=torch.float32)
    feat = [[ELEM.index(a.GetSymbol()) if a.GetSymbol() in ELEM else len(ELEM),
             min(a.GetDegree(), 6), min(max(a.GetFormalCharge(), -2), 2) + 2, int(a.GetIsAromatic()),
             HYB.index(a.GetHybridization()) if a.GetHybridization() in HYB else len(HYB),
             min(a.GetTotalNumHs(), 4), int(a.IsInRing()),
             min(int(a.GetChiralTag()), N_CHIRAL - 1)] for a in mol.GetAtoms()]
    n = len(feat)
    bond = torch.zeros(n, n, dtype=torch.long)
    stereo = torch.zeros(n, n, dtype=torch.long)
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bond[i, j] = bond[j, i] = BOND.get(b.GetBondType(), 1)
        stereo[i, j] = stereo[j, i] = min(int(b.GetStereo()), N_STEREO - 1)
    # Bonds in both directions, plus any other pair within 5 Å. The fully connected graph
    # averaged each bond together with every other atom, so the group was diluted by the degree.
    dist = torch.cdist(pos, pos)
    linked = ((bond > 0) | (dist <= 5)) & ~torch.eye(n, dtype=torch.bool)
    src, dst = linked.nonzero(as_tuple=True)
    return dict(pos=pos - pos.mean(0), feat=torch.tensor(feat, dtype=torch.long),
                edge=torch.stack([src, dst]),
                eattr=torch.stack([bond[src, dst], stereo[src, dst]], -1))

## 3. Protein graph

Each protein has one AlphaFold 3 prediction (monomer, MSA from UniRef90, model seed 1, one diffusion
sample). Only the Cα atom of each residue is used. `protein_ca.npz` stores all proteins back to back:

| Array | Shape | Content |
|---|---|---|
| `pid` | (P,) | protein id = md5(sequence)[:16] |
| `offset` | (P+1,) | protein *i* is rows `offset[i] : offset[i+1]` of the arrays below |
| `aa` | (R,) | one-letter residue type (`X` = non-standard) |
| `ca_xyz` | (R, 3) | Cα coordinates in Å |
| `plddt` | (R,) | AF3 per-residue confidence, 0–100 |

Each residue is connected to its peptide neighbours (i±1) and to every other residue within 8 Å.
The sequence separation `|i-j|` is binned into 8 classes as the edge feature. Proteins longer
than `max_res` keep their N-terminal `max_res` residues.

**Node features.** Three groups, each projected to `dim` and summed:

1. **Sequence window** — the residue types at i-4 … i+4 (clamped at the chain ends), concatenated. This
   is the local-motif information DrugBAN gets from its convolution kernels of width 3/6/9, and which a
   purely spatial graph would otherwise lose.
2. **Local backbone geometry** — Cα–Cα distances to i±1 and i±2, the pseudo bond angle at i, and the
   *signed* pseudo dihedral over (i-1, i, i+1, i+2). Distances and the angle are invariant under any
   rigid motion including reflection; the dihedral's sine **flips sign under reflection**, which is what
   lets the model tell a right-handed α-helix from an impossible left-handed one.
3. **pLDDT** — AF3's per-residue confidence, expanded on a radial basis rather than pushed through
   `Linear(1, dim)`.

In [ ]:
# ===================== 3. Protein: AF3 Ca trace -> graph =====================
AA = 'ACDEFGHIKLMNPQRSTVWY'           # index 20 = anything else
PROT_CUTOFF = 8.0                 # Ca–Ca contact distance, in Å; peptide bonds are always kept
SEP_BINS = [1, 2, 3, 4, 8, 16, 32]     # sequence separation |i-j| binned -> 8 classes
SEQ_WIN = 4                            # each residue sees the residue types at i-4 .. i+4
N_GEOM = 7                             # 4 distances + bond angle + 2 dihedral components


def load_protein_structures(path, keep=None):
    z = np.load(path)
    aa, xyz, plddt, off = z['aa'], z['ca_xyz'], z['plddt'], z['offset']
    return {pid: (''.join(aa[s:e]), xyz[s:e], plddt[s:e])
            for pid, s, e in zip(z['pid'], off[:-1], off[1:]) if keep is None or pid in keep}


def local_geometry(pos):
    """Rigid-motion-invariant description of the Ca trace around each residue.

    The last component (sine of the pseudo dihedral) changes sign under reflection, so the protein
    encoder can distinguish a right-handed helix from its mirror image. Everything else is invariant
    under reflection as well."""
    n = len(pos)
    i = torch.arange(n)
    at = lambda d: pos[(i + d).clamp(0, n - 1)]
    a, b, c, d = at(-1), pos, at(1), at(2)
    u, v, w = b - a, c - b, d - c
    dist = torch.stack([u.norm(dim=-1), v.norm(dim=-1),
                        (b - at(-2)).norm(dim=-1), (at(2) - b).norm(dim=-1)], -1) / 10
    un, vn, wn = [x / (x.norm(dim=-1, keepdim=True) + 1e-8) for x in (u, v, w)]
    nuv, nvw = torch.cross(un, vn, dim=-1), torch.cross(vn, wn, dim=-1)
    ang = torch.stack([(-un * vn).sum(-1),                      # cos of the pseudo bond angle at i
                       (nuv * nvw).sum(-1),                     # cos of the pseudo dihedral
                       (torch.cross(nuv, nvw, dim=-1) * vn).sum(-1)], -1)   # sin: sign flips on reflection
    return torch.cat([dist, ang], -1)


def protein_graph(aa, xyz, plddt, max_res=None):
    if max_res and len(aa) > max_res:        # keep the N-terminal max_res residues
        aa, xyz, plddt = aa[:max_res], xyz[:max_res], plddt[:max_res]
    pos = torch.tensor(np.ascontiguousarray(xyz), dtype=torch.float32)
    pos = pos - pos.mean(0)
    n = len(pos)
    # i±1 keeps the chain connected. Other edges are Ca pairs within 8 Å, so a residue no longer
    # aggregates neighbours 20 Å away the way a fixed 16-nearest graph did.
    dist = torch.cdist(pos, pos)
    chain = (torch.arange(n)[:, None] - torch.arange(n)[None, :]).abs() == 1
    linked = ((dist <= PROT_CUTOFF) | chain) & ~torch.eye(n, dtype=torch.bool)
    src, dst = linked.nonzero(as_tuple=True)
    sep = torch.bucketize((src - dst).abs(), torch.tensor(SEP_BINS))
    feat = torch.tensor([AA.index(a) if a in AA else 20 for a in aa])
    seq_idx = (torch.arange(n)[:, None] + torch.arange(-SEQ_WIN, SEQ_WIN + 1)).clamp(0, n - 1)
    return dict(pos=pos, feat=feat, plddt=torch.tensor(plddt, dtype=torch.float32) / 100,
                geom=local_geometry(pos), seq_idx=seq_idx,
                edge=torch.stack([src, dst]), eattr=sep)

## 4. Model

- `EGNNLayer`: messages are computed from the two node features, their distance and the edge feature;
  node features are updated from the messages (invariant) and coordinates are moved along the relative
  vectors (equivariant). Only the node features are passed on.
- `DrugEncoder` / `ProteinEncoder`: build the node features described above, then `drug_layers` /
  `prot_layers` EGNN layers.
- `BANLayer`, `FCNet`, `MLPDecoder`: DrugBAN's code, with **one change** — `forward` takes the padding
  masks and zeroes the padded positions after `v_net` / `q_net`. Without this, every padded position adds
  a constant to the fused feature (`FCNet` ends in `Linear + ReLU`, so zeros in do not give zeros out),
  and because the padding length is the batch maximum, the same pair's features depended on the rest of
  its batch. Set `CFG['mask_pad'] = False` to reproduce the old behaviour.

The cell ends with a self-check: a proper rotation plus translation must leave the prediction unchanged,
while a mirror image is expected to change it (the protein geometry features are chirality-aware).

In [ ]:
# ===================== 4. Model: EGNN encoders + DrugBAN bilinear attention =====================
# ---------- EGNN ----------
class RBF(nn.Module):
    def __init__(self, n=32, cutoff=20.0):
        super().__init__()
        self.register_buffer('mu', torch.linspace(0, cutoff, n))
        self.gamma = (n / cutoff) ** 2

    def forward(self, d):
        return torch.exp(-self.gamma * (d - self.mu) ** 2)


class EGNNLayer(nn.Module):
    """Messages depend only on distances, so the node update is invariant. Coordinates move along
    relative vectors only when `move_coords` is set. The drug encoder turns that off and keeps the
    input conformer's distances at every layer; the protein encoder leaves it on."""

    def __init__(self, dim, edim, n_rbf=32, move_coords=True):
        super().__init__()
        self.move_coords = move_coords
        self.rbf = RBF(n_rbf)
        self.msg = nn.Sequential(nn.Linear(2 * dim + n_rbf + edim, dim), nn.SiLU(), nn.Linear(dim, dim), nn.SiLU())
        self.gate = nn.Sequential(nn.Linear(dim, 1), nn.Sigmoid())
        if move_coords:
            self.coord = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, 1, bias=False))
            nn.init.xavier_uniform_(self.coord[-1].weight, gain=1e-3)
        self.upd = nn.Sequential(nn.Linear(2 * dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.norm = nn.LayerNorm(dim)

    def forward(self, h, x, edge, e):
        src, dst = edge
        rel = x[dst] - x[src]
        dist = rel.norm(dim=-1, keepdim=True)
        m = self.msg(torch.cat([h[dst], h[src], self.rbf(dist), e], -1))
        m = m * self.gate(m)
        # Degree in float32: bfloat16 has only 8 mantissa bits. Under autocast h, m and the
        # coordinates can each end up in a different dtype -- index_add_ does not promote, so every
        # accumulator and divisor below is cast explicitly.
        deg = torch.zeros(len(h), 1, device=h.device, dtype=dist.dtype).index_add_(
            0, dst, torch.ones_like(dist)).clamp(min=1)
        if self.move_coords:
            shift = (rel / (dist + 1) * torch.tanh(self.coord(m))).to(x.dtype)
            x = x + torch.zeros_like(x).index_add_(0, dst, shift) / deg.to(x.dtype)
        agg = torch.zeros_like(h).index_add_(0, dst, m.to(h.dtype)) / deg.to(h.dtype)
        h = self.norm(h + self.upd(torch.cat([h, agg], -1)))
        return h, x


class EGNN(nn.Module):
    def __init__(self, dim, edim, n_layers, move_coords=True):
        super().__init__()
        self.layers = nn.ModuleList([
            EGNNLayer(dim, edim, move_coords=move_coords) for _ in range(n_layers)])

    def forward(self, h, x, edge, e):
        for layer in self.layers:
            h, x = layer(h, x, edge, e)
        return h


class DrugEncoder(nn.Module):
    """Atom features -> EGNN on bonds plus pairs within 5 Å.

    Coordinates stay at the input conformer. The attention receives the EGNN state concatenated
    with the per-atom features from before message passing, so a functional group is still visible
    after the layers mix neighbours."""

    def __init__(self, dim, n_layers, emb=32, edim=24):
        super().__init__()
        self.emb = nn.ModuleList([nn.Embedding(v, emb) for v in ATOM_VOCAB])
        self.proj = nn.Sequential(nn.Linear(len(ATOM_VOCAB) * emb, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.bond = nn.Embedding(N_BOND, edim - edim // 3)
        self.stereo = nn.Embedding(N_STEREO, edim // 3)
        self.egnn = EGNN(dim, edim, n_layers, move_coords=False)
        self.out = nn.LayerNorm(dim)
        self.local = nn.LayerNorm(dim)

    def forward(self, g):
        h0 = self.proj(torch.cat([e(g['feat'][:, i]) for i, e in enumerate(self.emb)], -1))
        e = torch.cat([self.bond(g['eattr'][:, 0]), self.stereo(g['eattr'][:, 1])], -1)
        h = self.out(self.egnn(h0, g['pos'], g['edge'], e))
        return torch.cat([h, self.local(h0)], -1)


class ProteinEncoder(nn.Module):
    """Sequence window + backbone geometry + pLDDT -> EGNN over peptide bonds and 8 Å contacts.

    The attention receives the EGNN state concatenated with the per-residue features from before
    message passing, so the local sequence and geometry stay visible after neighbours are mixed."""

    def __init__(self, dim, n_layers, emb=32, edim=24, n_plddt=16):
        super().__init__()
        self.aa = nn.Embedding(21, emb)
        self.win = nn.Sequential(nn.Linear((2 * SEQ_WIN + 1) * emb, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.plddt_rbf = RBF(n_plddt, 1.0)
        self.extra = nn.Sequential(nn.Linear(N_GEOM + n_plddt, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.sep = nn.Embedding(len(SEP_BINS) + 1, edim)
        self.egnn = EGNN(dim, edim, n_layers)
        self.out = nn.LayerNorm(dim)
        self.local = nn.LayerNorm(dim)

    def forward(self, g):
        win = self.aa(g['feat'][g['seq_idx']]).flatten(1)            # (R, (2*SEQ_WIN+1) * emb)
        extra = torch.cat([g['geom'], self.plddt_rbf(g['plddt'][:, None])], -1)
        h0 = self.win(win) + self.extra(extra)
        h = self.out(self.egnn(h0, g['pos'], g['edge'], self.sep(g['eattr'])))
        return torch.cat([h, self.local(h0)], -1)


def to_dense(h, batch, idx, B):
    """Concatenated node features -> zero-padded (B, Nmax, D) + mask, for the bilinear attention."""
    out = h.new_zeros(B, int(idx.max()) + 1, h.size(-1))
    mask = torch.zeros(B, out.size(1), dtype=torch.bool, device=h.device)
    out[batch, idx] = h
    mask[batch, idx] = True
    return out, mask


# ---------- Bilinear attention network + decoder: DrugBAN's code ----------
# From DrugBAN (https://github.com/peizhenbai/DrugBAN, ban.py and models.py), MIT License, Copyright (c) 2022 peizhenbai.
# BANLayer / FCNet are themselves modified from ban-vqa (https://github.com/jnhwkim/ban-vqa).
# CHANGE vs upstream: forward() accepts v_mask / q_mask and zeroes the padded positions (marked below).
from torch.nn.utils.weight_norm import weight_norm


class BANLayer(nn.Module):
    def __init__(self, v_dim, q_dim, h_dim, h_out, act='ReLU', dropout=0.2, k=3):
        super(BANLayer, self).__init__()

        self.c = 32
        self.k = k
        self.v_dim = v_dim
        self.q_dim = q_dim
        self.h_dim = h_dim
        self.h_out = h_out

        self.v_net = FCNet([v_dim, h_dim * self.k], act=act, dropout=dropout)
        self.q_net = FCNet([q_dim, h_dim * self.k], act=act, dropout=dropout)
        # self.dropout = nn.Dropout(dropout[1])
        if 1 < k:
            self.p_net = nn.AvgPool1d(self.k, stride=self.k)

        if h_out <= self.c:
            self.h_mat = nn.Parameter(torch.Tensor(1, h_out, 1, h_dim * self.k).normal_())
            self.h_bias = nn.Parameter(torch.Tensor(1, h_out, 1, 1).normal_())
        else:
            self.h_net = weight_norm(nn.Linear(h_dim * self.k, h_out), dim=None)

        self.bn = nn.BatchNorm1d(h_dim)

    def attention_pooling(self, v, q, att_map):
        fusion_logits = torch.einsum('bvk,bvq,bqk->bk', (v, att_map, q))
        if 1 < self.k:
            fusion_logits = fusion_logits.unsqueeze(1)  # b x 1 x d
            fusion_logits = self.p_net(fusion_logits).squeeze(1) * self.k  # sum-pooling
        return fusion_logits

    def forward(self, v, q, softmax=False, v_mask=None, q_mask=None):
        v_num = v.size(1)
        q_num = q.size(1)
        if self.h_out <= self.c:
            v_ = self.v_net(v)
            q_ = self.q_net(q)
            # --- CHANGE: drop padded positions. FCNet ends in Linear+ReLU, so a zero row does not stay
            # zero, and every padded position would otherwise add a constant to the fused feature. ---
            if v_mask is not None:
                v_ = v_ * v_mask.unsqueeze(-1).to(v_.dtype)
            if q_mask is not None:
                q_ = q_ * q_mask.unsqueeze(-1).to(q_.dtype)
            att_maps = torch.einsum('xhyk,bvk,bqk->bhvq', (self.h_mat, v_, q_)) + self.h_bias
        else:
            v_ = self.v_net(v).transpose(1, 2).unsqueeze(3)
            q_ = self.q_net(q).transpose(1, 2).unsqueeze(2)
            d_ = torch.matmul(v_, q_)  # b x h_dim x v x q
            att_maps = self.h_net(d_.transpose(1, 2).transpose(2, 3))  # b x v x q x h_out
            att_maps = att_maps.transpose(2, 3).transpose(1, 2)  # b x h_out x v x q
        if softmax:
            p = nn.functional.softmax(att_maps.view(-1, self.h_out, v_num * q_num), 2)
            att_maps = p.view(-1, self.h_out, v_num, q_num)
        logits = self.attention_pooling(v_, q_, att_maps[:, 0, :, :])
        for i in range(1, self.h_out):
            logits_i = self.attention_pooling(v_, q_, att_maps[:, i, :, :])
            logits += logits_i
        logits = self.bn(logits)
        return logits, att_maps


class FCNet(nn.Module):
    """Simple class for non-linear fully connect network
    Modified from https://github.com/jnhwkim/ban-vqa/blob/master/fc.py
    """

    def __init__(self, dims, act='ReLU', dropout=0):
        super(FCNet, self).__init__()

        layers = []
        for i in range(len(dims) - 2):
            in_dim = dims[i]
            out_dim = dims[i + 1]
            if 0 < dropout:
                layers.append(nn.Dropout(dropout))
            layers.append(weight_norm(nn.Linear(in_dim, out_dim), dim=None))
            if '' != act:
                layers.append(getattr(nn, act)())
        if 0 < dropout:
            layers.append(nn.Dropout(dropout))
        layers.append(weight_norm(nn.Linear(dims[-2], dims[-1]), dim=None))
        if '' != act:
            layers.append(getattr(nn, act)())

        self.main = nn.Sequential(*layers)

    def forward(self, x):
        return self.main(x)


class MLPDecoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, binary=1):
        super(MLPDecoder, self).__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, out_dim)
        self.bn3 = nn.BatchNorm1d(out_dim)
        self.fc4 = nn.Linear(out_dim, binary)

    def forward(self, x):
        x = self.bn1(F.relu(self.fc1(x)))
        x = self.bn2(F.relu(self.fc2(x)))
        x = self.bn3(F.relu(self.fc3(x)))
        x = self.fc4(x)
        return x


class E3DTI(nn.Module):
    """EGNN drug / protein encoders + DrugBAN's bilinear attention and MLP decoder.
    BAN hyperparameters follow DrugBAN's config: h_dim = DECODER.IN_DIM = 256, h_out = BCN.HEADS = 2,
    k = 3, dropout 0.2, no softmax, weight norm on h_mat; decoder 256 -> 512 -> 512 -> 128 -> 1."""

    def __init__(self, dim=256, drug_layers=4, prot_layers=2, emb=32, edim=24,
                 ban_dim=256, heads=2, mlp_hidden=512, mlp_out=128, mask_pad=True):
        super().__init__()
        self.mask_pad = mask_pad
        self.drug = DrugEncoder(dim, drug_layers, emb, edim)
        self.prot = ProteinEncoder(dim, prot_layers, emb, edim)
        # Both encoders return concat(EGNN state, features from before message passing).
        self.bcn = weight_norm(BANLayer(v_dim=dim * 2, q_dim=dim * 2, h_dim=ban_dim, h_out=heads), name='h_mat', dim=None)
        self.mlp_classifier = MLPDecoder(ban_dim, mlp_hidden, mlp_out, binary=1)

    def forward(self, gd, gp):
        B = int(gd['n_graphs'])
        v_d, m_d = to_dense(self.drug(gd), gd['batch'], gd['idx'], B)
        v_p, m_p = to_dense(self.prot(gp), gp['batch'], gp['idx'], B)
        if not self.mask_pad:
            m_d = m_p = None
        f, att = self.bcn(v_d, v_p, v_mask=m_d, q_mask=m_p)
        return self.mlp_classifier(f).squeeze(-1), att


OFFSET_KEYS = {'edge': 1, 'seq_idx': 0}      # key -> dim to concatenate along; values are node indices


def collate_graphs(gs):
    """Merge a batch of graphs into one: concatenate nodes, offset every tensor that holds node
    indices, and record each node's graph (batch) and its position within that graph (idx)."""
    n = torch.tensor([len(g['pos']) for g in gs])
    off = torch.cumsum(n, 0) - n
    out = {}
    for k in gs[0]:
        if k in OFFSET_KEYS:
            out[k] = torch.cat([g[k] + o for g, o in zip(gs, off)], OFFSET_KEYS[k])
        else:
            out[k] = torch.cat([g[k] for g in gs])
    out['batch'] = torch.arange(len(gs)).repeat_interleave(n)
    out['idx'] = torch.cat([torch.arange(k) for k in n.tolist()])
    out['n_graphs'] = torch.tensor(len(gs))
    return out


# ---------- Self-check ----------
def _orthogonal(mirror=False):
    q = torch.linalg.qr(torch.randn(3, 3))[0]
    if (torch.det(q) < 0) != mirror:       # det > 0 = proper rotation, det < 0 = rotation + reflection
        q[:, 0] = -q[:, 0]
    return q.numpy()


torch.manual_seed(0); np.random.seed(0)
_mdl = E3DTI().eval()
_mol = Chem.AddHs(Chem.MolFromSmiles('C[C@H](N)C(=O)Nc1ccc(F)cc1'))
AllChem.EmbedMolecule(_mol, randomSeed=0)
_gd0 = drug_graph(Chem.RemoveHs(_mol))
_xyz = (np.random.RandomState(0).randn(40, 3) * 8).astype(np.float32)
_plddt = (np.random.RandomState(1).rand(40) * 100).astype(np.float32)
_aa = (AA * 2)[:40]


def _batch(mirror=False):
    R, t = _orthogonal(mirror), (np.random.randn(3) * 10).astype(np.float32)
    gd = {**_gd0, 'pos': torch.tensor(_gd0['pos'].numpy() @ R.T + t)}
    gp = protein_graph(_aa, _xyz @ R.T + t, _plddt)
    return collate_graphs([gd, gd]), collate_graphs([gp, gp])


def _logits(mirror=False):
    gd, gp = _batch(mirror)
    with torch.no_grad():
        return _mdl(gd, gp)[0]


_a, _b, _c = _logits(), _logits(), _logits(mirror=True)
# Mixed precision mixes dtypes inside the EGNN (index_add_ does not promote), so exercise it here on
# the CPU instead of discovering it on the GPU half an hour into a job.
_gd_chk, _gp_chk = _batch()
with torch.no_grad(), torch.autocast('cpu', dtype=torch.bfloat16):
    _ = _mdl(_gd_chk, _gp_chk)
print('bfloat16 forward pass    : OK')
print(f'rotation + translation : max logit change {(_a - _b).abs().max().item():.2e}   <- must be ~0')
print(f'mirror image           : max logit change {(_a - _c).abs().max().item():.2e}   '
      '<- deliberately non-zero (proteins are chiral)')
print(f'Model parameters: {sum(p.numel() for p in _mdl.parameters()) / 1e6:.2f} M '
      f"(drug {sum(p.numel() for p in _mdl.drug.parameters()) / 1e6:.2f} M, "
      f"protein {sum(p.numel() for p in _mdl.prot.parameters()) / 1e6:.2f} M)")

## 5. Load the split and build the graphs

`random` and `cold` come with DrugBAN's `train`, `val` and `test` files. `cluster` is DrugBAN's
cross-domain split: train on the source domain (`source_train`) and test on the target domain
(`target_test`). DrugBAN uses `target_train` without labels for domain adaptation, which this notebook
does not do, so that file is not used. There is no validation file for `cluster`, so 10% of
`source_train` is held out for early stopping.

The csv files on Hugging Face only contain pairs whose protein has an AF3 structure and whose compound
has a conformer (the 11 longest proteins and one compound, nitroprusside, were removed when the dataset
was built). The cell checks this anyway and prints how many pairs have both structures; it should be 100%.

**Batching.** With random batches of 32, the protein side is padded to 1759 residues on average while the
true mean is 540 — 226% of the work is padding. `LengthBatchSampler` shuffles the pairs, cuts them into
pools of `bucket_pool` batches, sorts each pool by protein length and shuffles the resulting batches, so
batches stay random from epoch to epoch while holding proteins of similar length.

In [ ]:
# ===================== 5. Data: pairs -> (drug graph, protein graph, label) =====================
split_dir = f"{DATA}/main/{CFG['dataset']}/{CFG['split']}"
if CFG['split'] == 'cluster':
    src = pd.read_csv(f'{split_dir}/source_train.csv')
    val = src.sample(frac=0.1, random_state=CFG['seed'])
    dfs = dict(train=src.drop(val.index), val=val, test=pd.read_csv(f'{split_dir}/target_test.csv'))
else:
    dfs = {s: pd.read_csv(f'{split_dir}/{s}.csv') for s in ['train', 'val', 'test']}

for df in dfs.values():
    df['did'], df['pid'] = df.SMILES.map(md5), df.Protein.map(md5)
need_d, need_p = set(pd.concat(dfs.values()).did), set(pd.concat(dfs.values()).pid)

t0 = time.time()
MOLS = load_drug_conformers(f'{DATA}/main/drug_sdf/drug_conformers.sdf.gz', keep=need_d)
STRUCT = load_protein_structures(f'{DATA}/main/protein_af3/protein_ca.npz', keep=need_p)
print(f'Loaded {len(MOLS)} of {len(need_d)} conformers and {len(STRUCT)} of {len(need_p)} '
      f'protein structures this split needs ({time.time() - t0:.0f}s)')

for s, df in dfs.items():
    ok = df.did.isin(MOLS.keys()) & df.pid.isin(STRUCT.keys())
    print(f'{s:5s}: {len(df)} pairs, {ok.sum()} with both structures ({ok.mean():.1%}), '
          f'positive rate {df.Y[ok].mean():.3f}')
    dfs[s] = df[ok].reset_index(drop=True)
if CFG['max_train_pairs']:                      # smoke test: cap every split, so graph building is fast too
    for s in list(dfs):
        dfs[s] = dfs[s].sample(min(CFG['max_train_pairs'], len(dfs[s])), random_state=0).reset_index(drop=True)
    print('smoke test: capped to', {s: len(d) for s, d in dfs.items()})

t0 = time.time()
all_df = pd.concat(dfs.values())
DRUG = {d: drug_graph(MOLS[d]) for d in set(all_df.did)}
PROT = {p: protein_graph(*STRUCT[p], max_res=CFG['max_res']) for p in set(all_df.pid)}
_res = np.array([len(g['pos']) for g in PROT.values()])
print(f'Built graphs in {time.time() - t0:.0f}s: {len(DRUG)} drugs '
      f'(mean {np.mean([len(g["pos"]) for g in DRUG.values()]):.0f} heavy atoms), '
      f'{len(PROT)} proteins (mean {_res.mean():.0f} residues, max {_res.max()}, '
      f'{(_res == CFG["max_res"]).sum()} truncated at max_res={CFG["max_res"]})')


# The RDKit molecules and the raw coordinate arrays are no longer needed once the graphs exist, and
# dropping them before the DataLoader workers fork saves a few GB per worker. Re-run this whole cell to
# get them back.
del MOLS, STRUCT
gc.collect()


class PairSet(torch.utils.data.Dataset):
    def __init__(self, df):
        self.d, self.p, self.y = df.did.tolist(), df.pid.tolist(), df.Y.astype(np.float32).tolist()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return DRUG[self.d[i]], PROT[self.p[i]], self.y[i]


def collate(batch):
    gd, gp, y = zip(*batch)
    return collate_graphs(gd), collate_graphs(gp), torch.tensor(y)


class LengthBatchSampler(torch.utils.data.Sampler):
    """Batches of pairs whose proteins have similar length, so zero padding stays small.

    Indices are shuffled, cut into pools of `pool` batches, sorted by protein length inside each pool,
    and the resulting batches are shuffled -- batches differ every epoch but stay length-homogeneous."""

    def __init__(self, lengths, batch_size, shuffle=True, drop_last=False, pool=32, seed=0):
        self.len = np.asarray(lengths)
        self.bs, self.shuffle, self.drop_last, self.pool, self.seed = batch_size, shuffle, drop_last, pool, seed
        self.epoch = 0

    def __iter__(self):
        idx = np.arange(len(self.len))
        if self.shuffle:
            self.epoch += 1
            np.random.default_rng(self.seed + self.epoch).shuffle(idx)
        chunk, batches = self.bs * self.pool, []
        for s in range(0, len(idx), chunk):
            part = idx[s:s + chunk]
            part = part[np.argsort(self.len[part], kind='stable')]
            batches += [part[i:i + self.bs].tolist() for i in range(0, len(part), self.bs)]
        if self.drop_last:
            batches = [b for b in batches if len(b) == self.bs]
        if self.shuffle:
            np.random.default_rng(self.seed + 10000 + self.epoch).shuffle(batches)
        return iter(batches)

    def __len__(self):
        full, rem = divmod(len(self.len), self.bs)
        return full if self.drop_last or rem == 0 else full + 1


def loader(s, shuffle):
    ds, kw = PairSet(dfs[s]), dict(collate_fn=collate, num_workers=CFG['workers'],
                                   persistent_workers=CFG['workers'] > 0)
    if CFG['bucket']:
        sampler = LengthBatchSampler([len(PROT[p]['pos']) for p in dfs[s].pid], CFG['batch'],
                                     shuffle=shuffle, drop_last=shuffle,
                                     pool=CFG['bucket_pool'], seed=CFG['seed'])
        return torch.utils.data.DataLoader(ds, batch_sampler=sampler, **kw)
    return torch.utils.data.DataLoader(ds, batch_size=CFG['batch'], shuffle=shuffle, drop_last=shuffle, **kw)


dl = {s: loader(s, s == 'train') for s in ['train', 'val', 'test']}
_pad = [max(len(PROT[p]['pos']) for p in dfs['train'].pid[b]) for b in list(iter(dl['train'].batch_sampler))[:200]] \
    if CFG['bucket'] else None
if _pad:
    _true = np.mean([len(PROT[p]['pos']) for p in dfs['train'].pid])
    print(f'Batches of {CFG["batch"]}: padded to {np.mean(_pad):.0f} residues on average vs '
          f'{_true:.0f} true mean -> {np.mean(_pad) / _true - 1:.0%} of the compute is padding')

## 6. Training

AdamW with one warm-up epoch and a cosine learning-rate decay, binary cross-entropy loss. After every
epoch the model is scored on the validation set; the epoch with the best validation AUROC is kept
(`best.pt`), and training stops after `patience` epochs without improvement.

Mixed precision is on by default (`amp`): bfloat16 where the GPU supports it, otherwise float16 with a
gradient scaler. It changes only speed and numerics, not the model.

In [ ]:
# ===================== 6. Training =====================
from tqdm.auto import tqdm

model = E3DTI(CFG['dim'], CFG['drug_layers'], CFG['prot_layers'], CFG['emb'], CFG['edim'],
              mask_pad=CFG['mask_pad']).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
total, warm = CFG['epochs'] * len(dl['train']), CFG['warmup_epochs'] * len(dl['train'])
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda t: min(1, (t + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(t, total) / total)))
to_dev = lambda g: {k: v.to(dev, non_blocking=True) for k, v in g.items()}

use_amp = bool(CFG['amp']) and dev.type == 'cuda'
amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=use_amp and amp_dtype is torch.float16)
autocast = lambda: torch.autocast('cuda', dtype=amp_dtype, enabled=use_amp)


@torch.no_grad()
def predict(net, loader):
    net.eval()
    ys, ps = [], []
    for gd, gp, y in loader:
        with autocast():
            logit, _ = net(to_dev(gd), to_dev(gp))
        ys.append(y.numpy()); ps.append(torch.sigmoid(logit.float()).cpu().numpy())
    y, p = np.concatenate(ys), np.concatenate(ps)
    loss = F.binary_cross_entropy(torch.tensor(p).clamp(1e-7, 1 - 1e-7), torch.tensor(y)).item()
    return y, p, dict(auroc=roc_auc_score(y, p), auprc=average_precision_score(y, p), loss=loss)


print(f"Training on {dev}"
      f"{' (' + torch.cuda.get_device_name(0) + ')' if dev.type == 'cuda' else ''}"
      f", mixed precision {amp_dtype if use_amp else 'off'}: up to {CFG['epochs']} epochs, "
      f"{len(dl['train'])} batches per epoch. One line per epoch; the bar shows progress inside it.",
      flush=True)
history, best, best_state, bad = [], -1, None, 0
for ep in range(1, CFG['epochs'] + 1):
    model.train()
    t0, run_loss = time.time(), 0.0
    for gd, gp, y in tqdm(dl['train'], desc=f"epoch {ep}/{CFG['epochs']}", leave=False):
        with autocast():
            logit, _ = model(to_dev(gd), to_dev(gp))
            loss = F.binary_cross_entropy_with_logits(logit.float(), y.to(dev))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        run_loss += loss.item()
    _, _, val = predict(model, dl['val'])
    history.append(dict(epoch=ep, train_loss=run_loss / len(dl['train']), **{f'val_{k}': v for k, v in val.items()}))
    flag = ''
    if val['auroc'] > best:
        best, best_state, bad, flag = val['auroc'], copy.deepcopy(model.state_dict()), 0, ' *'
        torch.save({'model': best_state, 'cfg': CFG, 'epoch': ep}, f'{OUT}/best.pt')
    else:
        bad += 1
    print(f"ep {ep:3d}/{CFG['epochs']} | train loss {history[-1]['train_loss']:.4f} | "
          f"val loss {val['loss']:.4f} AUROC {val['auroc']:.4f} AUPRC {val['auprc']:.4f} | "
          f"{time.time() - t0:.0f}s"
          f"{f' | peak {torch.cuda.max_memory_allocated() / 2 ** 30:.1f} GB' if dev.type == 'cuda' else ''}"
          f"{flag}", flush=True)
    pd.DataFrame(history).to_csv(f'{OUT}/history.csv', index=False)
    if bad >= CFG['patience']:
        print(f'Validation AUROC has not improved for {bad} epochs; stopping early')
        break

## 7. Test

The best epoch is evaluated on the test set. AUROC and AUPRC need no threshold. For accuracy,
sensitivity, specificity, precision and F1, the threshold that maximizes F1 on the **validation** set is
used, so the test labels are never looked at.

For reference, DrugBAN reports (mean ± std over 5 runs) AUROC 0.903 ± 0.005 / AUPRC 0.902 ± 0.004 on
BioSNAP random and AUROC 0.960 ± 0.001 / AUPRC 0.948 ± 0.002 on BindingDB random; on the cross-domain
cluster splits it reports AUROC 0.654 ± 0.023 (BioSNAP) and 0.575 ± 0.025 (BindingDB) without domain
adaptation. A single run of this notebook will scatter around its own mean, so differences of ~0.01 are
not meaningful.

In [ ]:
# ===================== 7. Test (best validation epoch) =====================
model.load_state_dict(best_state)
yv, pv, val = predict(model, dl['val'])
yt, pt, test = predict(model, dl['test'])
prec, rec, thr = precision_recall_curve(yv, pv)
th = float(thr[np.argmax((2 * prec * rec / (prec + rec + 1e-12))[:-1])])
tn, fp, fn, tp = confusion_matrix(yt, pt >= th, labels=[0, 1]).ravel()
test.update(threshold=th, accuracy=(tp + tn) / len(yt), sensitivity=tp / max(tp + fn, 1),
            specificity=tn / max(tn + fp, 1), precision=tp / max(tp + fp, 1), f1=2 * tp / max(2 * tp + fp + fn, 1))
res = dict(cfg=CFG, best_epoch=int(np.argmax([h['val_auroc'] for h in history]) + 1),
           n_pairs={s: len(d) for s, d in dfs.items()}, val=val, test=test)
json.dump(res, open(f'{OUT}/results.json', 'w'), indent=1, default=float)
print(f"Best epoch {res['best_epoch']} | validation AUROC {val['auroc']:.4f} AUPRC {val['auprc']:.4f}")
print('Test: ' + ', '.join(f'{k} {v:.4f}' for k, v in test.items()))
print(f'Results saved to {OUT}/')